<h4>1. Load Data</h4>

In [1]:
import pandas as pd
import joblib
from pathlib import Path

In [2]:
#Path setup
base_dir = Path("..")
data_dir = base_dir / "data" / "processed"
model_dir = base_dir / "models"
preprocessing_dir = base_dir / "data" / "processed" / "preprocessing"


# load data satelit
df_predict = pd.read_csv(data_dir / "jakarta_features_1km_2023_2025.csv")



print(df_predict.shape)
df_predict.head()

(658, 8)


,lat,lon,s5p_co,s5p_no2,s5p_o3,s5p_so2,modis_aod_047,viirs_ntl
0,-6.364564,106.886044,0.034549,0.000085,0.116771,0.000083,374.666667,25.751492
1,-6.364564,106.895027,0.034534,0.000083,0.116759,0.000068,379.333333,20.227253
2,-6.364564,106.912994,0.034301,0.000081,0.116754,0.000069,380.333333,25.102145
3,-6.355581,106.796213,0.034512,0.000083,0.116755,0.000083,402.833333,31.692071
4,-6.355581,106.805196,0.034527,0.000087,0.116757,0.000098,402.750000,33.878588


<h4>1.1 Pisahkan Data</h4>

In [3]:
df_metadata = df_predict[["lat", "lon"]].copy()

df_metadata

,lat,lon
0,-6.364564,106.886044
1,-6.364564,106.895027
2,-6.364564,106.912994
3,-6.355581,106.796213
4,-6.355581,106.805196
...,...,...
653,-6.095069,106.742314
654,-6.095069,106.751297
655,-6.086086,106.733330
656,-6.086086,106.751297


<h4>1.2 Rename Kolom samakan</h4>

In [4]:
rename_map = {
    "modis_aod_047": "modis_MODIS_AOD_047",
    "viirs_ntl": "viirs_VIIRS_NTL"
}

df_predict = df_predict.rename(columns=rename_map)
print("Kolom setelah rename:", df_predict.columns.tolist())

Kolom setelah rename: ['lat', 'lon', 's5p_co', 's5p_no2', 's5p_o3', 's5p_so2', 'modis_MODIS_AOD_047', 'viirs_VIIRS_NTL']


In [5]:
feature_cols = [
    "s5p_co",
    "s5p_no2",
    "s5p_o3",
    "s5p_so2",
    "modis_MODIS_AOD_047",
    "viirs_VIIRS_NTL"
]
X_predict = df_predict[feature_cols].copy()
X_predict.head()

,s5p_co,s5p_no2,s5p_o3,s5p_so2,modis_MODIS_AOD_047,viirs_VIIRS_NTL
0,0.034549,0.000085,0.116771,0.000083,374.666667,25.751492
1,0.034534,0.000083,0.116759,0.000068,379.333333,20.227253
2,0.034301,0.000081,0.116754,0.000069,380.333333,25.102145
3,0.034512,0.000083,0.116755,0.000083,402.833333,31.692071
4,0.034527,0.000087,0.116757,0.000098,402.750000,33.878588


<h4>1.2 Load object trained</h4>

In [6]:
# Load trained objects
knn_imputer = joblib.load(preprocessing_dir / "knn_imputer.pkl")
robust_scaler = joblib.load(preprocessing_dir / "robust_scaler.pkl")
xgb_model = joblib.load(model_dir / "xgb_model_test.pkl")

<h3>2. Transform Data</h3>

In [7]:
#KNN Imputation
X_imputed = knn_imputer.transform(X_predict)

# Scaler/Rovust Scaling
X_scaled = robust_scaler.transform(X_imputed)

C:\Users\USER\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but RobustScaler was fitted with feature names
  warnings.warn(


<h4>2.1 Pred Kelas Kualitas Udara</h4>

In [9]:
y_pred = xgb_model.predict(X_scaled)
y_proba = xgb_model.predict_proba(X_scaled)


<h3>3. Gabung prediksi dengan metadata</h3>

In [10]:
df_result = df_metadata.copy()
df_result["predicted_class"] = y_pred

<h3>4. Save Data</h3>

In [11]:
out_path = data_dir / "prediction_results.csv"
df_result.to_csv(out_path, index=False)
print("Saved:", out_path)

Saved: ..\data\processed\prediction_results.csv


<h3>5. Label</h3>

In [12]:
class_map = {
    0 : "GOOD",
    1 : "MEDIUM",
    2 : "UNHEALTHY"
}

df_result["predicted_label"] = df_result["predicted_class"].map(class_map)

#check
print(df_result[["predicted_class", "predicted_label"]].head())

#save output
out_path = data_dir / "predicted_label.csv"
df_result.to_csv(out_path, index=False)

print("Saved to:", out_path)

   predicted_class predicted_label
0                1          MEDIUM
1                1          MEDIUM
2                2       UNHEALTHY
3                2       UNHEALTHY
4                2       UNHEALTHY
Saved to: ..\data\processed\predicted_label.csv


In [13]:
df_result["predicted_label"].value_counts()


predicted_label
MEDIUM       439
UNHEALTHY    141
GOOD          78
Name: count, dtype: int64